# Measurement_Findings — graph-fed consumer

Builds `sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx` for
the measurement sub-type of Findings. Two sheets:

| Sheet | Grain | Rows (expected) | Source |
|---|---|---|---|
| `Test_Identity` | TESTCD | 334 | `SDTM_Test_Identity.xlsx` widened to measurement scope; COSMoS coverage flags from `DSS_View.xlsx` |
| `Measurement_Specs` | DSS | 128 | `DSS_View.xlsx` filtered to VS, MK |

## Scope

Subject-level measurements without specimen decomposition. Behavioural rationale
in `cosmos-bc-dss/docs/COSMoS_Behavioural_Analysis.md`.

**In scope:** VS, MK, CV.
**Excluded:** EG (all BCs marked `Qualitative` despite ECG being inherently
quantitative; units present on a sizeable share — pending clarification).

CV is in scope but has no published DSSs — appears only in `Test_Identity`.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `consumer-bases/interim/DSS_View.xlsx` | consumer-bases | `Test_Identity`, `Measurement_Specs` |
| `sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | `Test Codes` |
| `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | `Domains` |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | `CodelistTerms` (for `Allowed_Units` expansion) |

## Output

`sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx`

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

print('MEASUREMENT_FINDINGS — graph-fed consumer')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

MEASUREMENT_FINDINGS — graph-fed consumer
Run: 2026-08-23 19:06


In [2]:
BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DSS_VIEW_FILE = REPO_ROOT / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'

OUTPUT_DIR = BASE_DIR / 'machine_actionable'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / 'Measurement_Findings.xlsx'

# Measurement-scope domains. Behavioural exclusions:
#   EG — all BCs marked Qualitative despite ECG being inherently quantitative;
#        18 DSSs have units despite the Qualitative scale. Pending clarification.
SCOPE_DOMAINS = ['VS', 'MK', 'CV']

# Allowed_Units expansion: expand ORRESU_codelist into permissible-value list
# only when the codelist has at most this many terms. UNIT (~950 terms) is too broad
# for per-row expansion; narrower sub-codelists like VSRESU (~29 terms) are useful.
ALLOWED_UNITS_THRESHOLD = 50

for f, label in [
    (DSS_VIEW_FILE, 'DSS_View'),
    (TEST_IDENTITY_FILE, 'SDTM_Test_Identity'),
    (DOMAIN_META_FILE, 'SDTM_Domain_Metadata'),
    (GRAPH_CT_FILE, 'COSMoS_Graph_CT'),
]:
    if not f.exists():
        raise FileNotFoundError(f'{label} not found: {f}')
    print(f'  {label}: {f.relative_to(REPO_ROOT)}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print(f'  Scope:  {SCOPE_DOMAINS}')
print(f'  Allowed_Units threshold: {ALLOWED_UNITS_THRESHOLD} terms')

  DSS_View: consumer-bases/interim/DSS_View.xlsx
  SDTM_Test_Identity: sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  SDTM_Domain_Metadata: sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  COSMoS_Graph_CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx
  Scope:  ['VS', 'MK', 'CV']
  Allowed_Units threshold: 50 terms


## 2. Load inputs

In [3]:
view_ti = pd.read_excel(DSS_VIEW_FILE, sheet_name='Test_Identity', dtype=str).fillna('')
view_ms = pd.read_excel(DSS_VIEW_FILE, sheet_name='Measurement_Specs', dtype=str).fillna('')
ref_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
ct_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str).fillna('')

print(f'DSS_View Test_Identity:     {len(view_ti):>6,} rows')
print(f'DSS_View Measurement_Specs: {len(view_ms):>6,} rows')
print(f'SDTM_Test_Identity:         {len(ref_tests):>6,} rows')
print(f'SDTM_Domain_Metadata:       {len(domain_meta):>6,} rows')
print(f'COSMoS CodelistTerms:       {len(ct_terms):>6,} rows')

DSS_View Test_Identity:        829 rows
DSS_View Measurement_Specs:  1,475 rows
SDTM_Test_Identity:          5,885 rows
SDTM_Domain_Metadata:           57 rows
COSMoS CodelistTerms:       17,607 rows


## 3. Build Test_Identity

Same shape as `Specimen_Findings.Test_Identity`. Universe is the wider
`SDTM_Test_Identity` filtered to measurement scope. COSMoS coverage flags
aggregated from scope DSSs only.

In [4]:
def in_scope(domains_str):
    if not domains_str:
        return False
    doms = {d.strip() for d in domains_str.split(';')}
    return bool(doms & set(SCOPE_DOMAINS))

ref_in_scope = ref_tests[ref_tests['SDTM_Domains'].apply(in_scope)].copy()
print(f"Measurement-scoped TESTCDs: {len(ref_in_scope):,}")

def filter_to_scope(domains_str):
    doms = sorted({d.strip() for d in domains_str.split(';') if d.strip() in SCOPE_DOMAINS})
    return '; '.join(doms)

ref_in_scope['In_Scope_Domains'] = ref_in_scope['SDTM_Domains'].apply(filter_to_scope)

print('Per-domain TESTCD count:')
for d in SCOPE_DOMAINS:
    n = ref_in_scope['SDTM_Domains'].apply(lambda s: d in [x.strip() for x in s.split(';')]).sum()
    print(f'  {d}: {n:>5,}')

Measurement-scoped TESTCDs: 334
Per-domain TESTCD count:
  VS:    76
  MK:    69
  CV:   191


In [5]:
view_ms_scope = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Measurement-scope DSS rows in DSS_View: {len(view_ms_scope):,}")

def _join_unique(s):
    return '; '.join(sorted({v for v in s if v}))

cosmos_cov = view_ms_scope.groupby('TESTCD_value', sort=True).agg(
    DSS_Count=('ds_id', 'nunique'),
    BC_Count=('bc_id', 'nunique'),
    DS_Codes=('ds_id', _join_unique),
    BC_IDs=('bc_id', _join_unique),
).reset_index().rename(columns={'TESTCD_value': 'TESTCD'})

cosmos_cov['DSS_Count'] = cosmos_cov['DSS_Count'].astype(str)
cosmos_cov['BC_Count'] = cosmos_cov['BC_Count'].astype(str)
print(f"COSMoS-pinned TESTCDs in scope: {len(cosmos_cov):,}")

flags = view_ti[['TESTCD', 'NCIt_Code_Conflict', 'NCIt_Reference_Disagree']].copy()

Measurement-scope DSS rows in DSS_View: 128
COSMoS-pinned TESTCDs in scope: 123


In [6]:
ti = ref_in_scope.copy()
ti['Has_DSS'] = ti['TESTCD'].isin(cosmos_cov['TESTCD']).map({True: 'Yes', False: 'No'})
ti = ti.merge(cosmos_cov, on='TESTCD', how='left').fillna('')
ti.loc[ti['DSS_Count'] == '', 'DSS_Count'] = '0'
ti.loc[ti['BC_Count'] == '', 'BC_Count'] = '0'
ti = ti.merge(flags, on='TESTCD', how='left').fillna('')

TI_COLS = [
    'TESTCD', 'NCIt_Code', 'In_Scope_Domains', 'SDTM_Domains',
    'TEST', 'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition',
    'UMLS_CUI', 'NCIm_CUI',
    'Has_DSS', 'DSS_Count', 'BC_Count', 'DS_Codes', 'BC_IDs',
    'NCIt_Code_Conflict', 'NCIt_Reference_Disagree',
]

missing = [c for c in TI_COLS if c not in ti.columns]
if missing:
    raise RuntimeError(f'Missing expected Test_Identity columns: {missing}')

ti_final = ti[TI_COLS].copy()
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")
print(f"  COSMoS-pinned (Has_DSS=Yes): {(ti_final['Has_DSS'] == 'Yes').sum():,}")
print(f"  Coverage gap (Has_DSS=No):   {(ti_final['Has_DSS'] == 'No').sum():,}")

Test_Identity: 334 rows x 17 cols
  COSMoS-pinned (Has_DSS=Yes): 123
  Coverage gap (Has_DSS=No):   211


## 4. Build Measurement_Specs

Filter to scope domains. Add `Observation_Class`, `LOINC_BC` (coalesce of two
source URI variants), `LOINC_DSS` (rename of `LOINC_value`), and `Allowed_Units`
(conditional expansion of `ORRESU_codelist` against `CodelistTerms` when the
codelist has ≤ ALLOWED_UNITS_THRESHOLD terms).

Slot block columns follow the SLOT_SCHEMA hybrid: core slots always emitted
(in declared order), optional slots emitted only when firing. A "slot" covers
both pinned variables (single value) and value_list-restricted variables (set
of allowed values) — mutually exclusive in COSMoS source, see DSS_View ReadMe.

Validation checks (TESTCD_NCIt ↔ BC NCIt consistency, Quantitative-without-units,
etc.) belong in a separate validation step, not in this consumer projection.
See `cosmos-bc-dss/notebooks/COSMoS_BC_DSS_Validate.ipynb` for the legacy QC pattern.

In [7]:
ms = view_ms[view_ms['domain'].isin(SCOPE_DOMAINS)].copy()
print(f"Measurement_Specs rows in scope: {len(ms):,}")
print(f"  Domains present:           {sorted(ms['domain'].unique())}")
print(f"  Domains with no DSSs yet:  {sorted(set(SCOPE_DOMAINS) - set(ms['domain'].unique()))}")

Measurement_Specs rows in scope: 128
  Domains present:           ['MK', 'VS']
  Domains with no DSSs yet:  ['CV']


In [8]:
# Observation_Class join.
ms = ms.merge(
    domain_meta[['Domain', 'Observation_Class']].rename(columns={'Domain': 'domain'}),
    on='domain', how='left',
  ).fillna('')

# LOINC_BC: coalesce the two source URI variants.
LOINC_BC_SOURCES = ['Coding_http://loinc.org/', 'Coding_https://loinc.org']
def _coalesce_loinc(row):
    parts = set()
    for col in LOINC_BC_SOURCES:
        v = row.get(col, '')
        if v:
            for p in v.split(';'):
                p = p.strip()
                if p:
                    parts.add(p)
    return '; '.join(sorted(parts))

ms['LOINC_BC'] = ms.apply(_coalesce_loinc, axis=1)
print(f'LOINC_BC populated rows: {(ms["LOINC_BC"] != "").sum()}')

LOINC_BC populated rows: 35


In [9]:
# Allowed_Units: conditional expansion of ORRESU_codelist when the codelist is narrow
# enough that per-row expansion is signal, not noise.
codelist_size = ct_terms.groupby('codelist_submission_value')['term_submission_value'].nunique()
narrow_codelists = set(codelist_size[codelist_size <= ALLOWED_UNITS_THRESHOLD].index)
print(f'Codelists with <= {ALLOWED_UNITS_THRESHOLD} terms: {len(narrow_codelists):,}')

terms_by_codelist = {}
for cl, group in ct_terms.groupby('codelist_submission_value'):
    if cl in narrow_codelists:
        terms = sorted({t for t in group['term_submission_value'] if t})
        terms_by_codelist[cl] = '; '.join(terms)

ms['Allowed_Units'] = ms['ORRESU_codelist'].map(terms_by_codelist).fillna('')
expanded = (ms['Allowed_Units'] != '').sum()
print(f'Rows with Allowed_Units expansion: {expanded:,}')
if expanded:
    fired_codelists = sorted({cl for cl in ms['ORRESU_codelist'] if cl in narrow_codelists})
    print(f'  Fired codelists: {fired_codelists}')

Codelists with <= 50 terms: 250


Rows with Allowed_Units expansion: 7
  Fired codelists: ['VSRESU']


In [10]:
# Rename DSS_View link-key columns for join symmetry with Test_Identity.
ms = ms.rename(columns={
    'TESTCD_value': 'TESTCD',
    'TEST_value':   'TEST',
    'TESTCD_ncit':  'TESTCD_NCIt',
    'LOINC_value':  'LOINC_DSS',
})

In [11]:
# SLOT_SCHEMA declares which slot remainders this consumer emits, classified by policy:
#   core     — column always emitted, even when no row fires (empty column).
#              Reserved for first-order structural slots (decomposition axes).
#   optional — column emitted only when at least one row fires.
#              Includes qualifier slots (units, location, laterality, position,
#              evaluator, evaluator id, test detail) and result-vocabulary slots
#              (ORRES, STRESC, STRESN — conditional on result_scales being
#              categorical), whose presence is a consequence of other columns.
#   exclude  — slot handled elsewhere (link-key, LOINC, or N/A for this sub-type).
# A "slot" covers both pinned variables (assigned_term_value/_ncit) and
# value_list-restricted variables. Pin and value_list are mutually exclusive
# in COSMoS source — see DSS_View ReadMe. Drift detection: slots firing in
# source but not declared get a WARN at runtime.
# SPEC is explicitly excluded — measurement is not specimen-decomposed by design.
SLOT_SCHEMA = {
    'core':     ['METHOD'],
    'optional': ['ORRES', 'STRESC', 'STRESN', 'ORRESU', 'STRESU',
                 'LOC', 'LAT', 'POS', 'EVAL', 'EVALID', 'CAT', 'TSTDTL'],
    'exclude':  ['TESTCD', 'TEST', 'LOINC', 'SPEC'],
}

SLOT_SUFFIXES = ('_value', '_ncit', '_codelist', '_value_list', '_value_list_ncit')


def fires(remainder):
    for suffix in SLOT_SUFFIXES:
        col = f'{remainder}{suffix}'
        if col in ms.columns and (ms[col] != '').any():
            return True
    return False


declared = set(SLOT_SCHEMA['core']) | set(SLOT_SCHEMA['optional']) | set(SLOT_SCHEMA['exclude'])

# Candidate remainders: any slot with a _value or _value_list column in source
all_candidate_remainders = sorted(
    {c.rsplit('_value_list_ncit', 1)[0] for c in ms.columns if c.endswith('_value_list_ncit')} |
    {c.rsplit('_value_list', 1)[0] for c in ms.columns if c.endswith('_value_list')} |
    {c.rsplit('_value', 1)[0] for c in ms.columns if c.endswith('_value') and not c.endswith('_value_list')}
)

slots_to_emit = []
notes = []

# Core: always emit, in declared order. Warn if no columns present in source.
for r in SLOT_SCHEMA['core']:
    if f'{r}_value' not in ms.columns and f'{r}_value_list' not in ms.columns:
        notes.append(f"WARN: core slot '{r}' has no columns in DSS_View — schema likely needs updating.")
        continue
    slots_to_emit.append(r)
    if not fires(r):
        notes.append(f"INFO: core slot '{r}' not firing on this scope — schema preserved, column will be empty.")

# Optional: emit only when firing, in declared order.
for r in SLOT_SCHEMA['optional']:
    if fires(r):
        slots_to_emit.append(r)

# Undeclared firing slots: WARN and emit at end (drift detection).
for r in all_candidate_remainders:
    if r in declared:
        continue
    if fires(r):
        col = f'{r}_value' if f'{r}_value' in ms.columns else f'{r}_value_list'
        n_rows = (ms[col] != '').sum()
        notes.append(f"WARN: undeclared slot '{r}' firing on {n_rows} rows — add to SLOT_SCHEMA core/optional/exclude.")
        slots_to_emit.append(r)

for note in notes:
    print(note)
if notes:
    print()

print(f"Slots to emit ({len(slots_to_emit)}, in order): {slots_to_emit}")
print(f"  Core (always present):  {[r for r in slots_to_emit if r in SLOT_SCHEMA['core']]}")
print(f"  Optional (firing-only): {[r for r in slots_to_emit if r in SLOT_SCHEMA['optional']]}")
print(f"  Undeclared (firing):    {[r for r in slots_to_emit if r not in declared]}")


Slots to emit (11, in order): ['METHOD', 'ORRES', 'STRESC', 'STRESN', 'ORRESU', 'STRESU', 'LOC', 'LAT', 'POS', 'EVAL', 'EVALID']
  Core (always present):  ['METHOD']
  Optional (firing-only): ['ORRES', 'STRESC', 'STRESN', 'ORRESU', 'STRESU', 'LOC', 'LAT', 'POS', 'EVAL', 'EVALID']
  Undeclared (firing):    []


In [12]:
# Assemble final column order — keys, test identity, BC identity, DSS identity,
# firing-pin triplets, LOINC pair, Allowed_Units, conflict flag.
identity_cols = [
    'domain', 'Observation_Class', 'ds_id', 'bc_id', 'TESTCD',
    'TEST', 'TESTCD_NCIt',
    'bc_short_name', 'bc_definition', 'bc_categories', 'bc_parent_label',
    'bc_hierarchy_path', 'bc_type', 'result_scales', 'bc_ncit_code',
    'ds_short_name', 'sdtmig_start_version', 'sdtmig_end_version',
    'package_date', 'source',
]

slot_cols = []
for r in slots_to_emit:
    for suffix in SLOT_SUFFIXES:
        col = f'{r}{suffix}'
        if col in ms.columns:
            slot_cols.append(col)

loinc_cols = ['LOINC_DSS', 'LOINC_BC']
tail_cols = ['Allowed_Units']

MS_COLS = identity_cols + slot_cols + loinc_cols + tail_cols

missing = [c for c in MS_COLS if c not in ms.columns]
if missing:
    raise RuntimeError(f'Missing expected Measurement_Specs columns: {missing}')

ms_final = ms[MS_COLS].copy()
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")
print(f"  Identity: {len(identity_cols)}  Slots: {len(slot_cols)}  LOINC: {len(loinc_cols)}  Tail: {len(tail_cols)}")

Measurement_Specs: 128 rows x 66 cols
  Identity: 20  Slots: 43  LOINC: 2  Tail: 1


## 5. Write workbook

Three sheets: `ReadMe`, `Test_Identity`, `Measurement_Specs`. Yellow-layout
colour convention.

In [13]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER  = PatternFill('solid', fgColor='548235')
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')
GREY_HEADER   = PatternFill('solid', fgColor='808080')


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'

print('Style helpers ready.')

Style helpers ready.


In [14]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Measurement_Findings — graph-fed consumer', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    ('Notebook: sdtm-findings-graph/notebooks/Measurement_Findings.ipynb', readme_font),
    ('Track:    sdtm-findings-graph (graph-fed Findings consumer)', readme_font),
    ('Inputs:', readme_font),
    ('  consumer-bases/interim/DSS_View.xlsx (Test_Identity, Measurement_Specs)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx (Test Codes)', readme_font),
    ('  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains)', readme_font),
    ('  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (CodelistTerms — for Allowed_Units)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Subject-level measurements without specimen decomposition.', readme_font),
    ('In scope: VS, MK, CV.', readme_font),
    ('Excluded: EG (all BCs Qualitative with units present — pending clarification).', readme_font),
    ('Behavioural rationale: cosmos-bc-dss/docs/COSMoS_Behavioural_Analysis.md', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD scoped to measurement domains. Universe', readme_font),
    ('  is the wider SDTM_Test_Identity (NCI EVS), filtered to scope. Has_DSS flags', readme_font),
    ('  whether COSMoS pins the TESTCD. DSS_Count / BC_Count / DS_Codes / BC_IDs', readme_font),
    ('  are aggregated over scope DSSs only.', readme_font),
    ('Measurement_Specs — one row per DSS in VS, MK. CV is in scope but has no', readme_font),
    ('  DSSs yet — it appears only in Test_Identity. Slot columns included only', readme_font),
    ('  when the slot actually fires on the scope subset — pin or value_list (data-driven shape).', readme_font),
    ('', None),
    ('CONSUMER-OWED DECISIONS APPLIED', section_font),
    ('LOINC: surfaced at both grains. LOINC_DSS = pinned LOINC at the DSS-Variable', readme_font),
    ('  level (empty for measurement — no DSS-grain LOINC pinned). LOINC_BC =', readme_font),
    ('  pinned LOINC at the BC level, coalesced across the two source URI variants', readme_font),
    ('  (graph projects the source URI inconsistency verbatim — see', readme_font),
    ('  cosmos-graph/docs/COSMoS_Graph.md).', readme_font),
    ('Allowed_Units expansion: applied. ORRESU_codelist is expanded against', readme_font),
    (f'  CodelistTerms only when the codelist has <= {{ALLOWED_UNITS_THRESHOLD}} terms — narrow enough'.replace('{{ALLOWED_UNITS_THRESHOLD}}', str(50)), readme_font),
    ('  to be readable per-row. Fires for VSRESU (~29 terms); does not fire for', readme_font),
    ('  the master UNIT codelist (~950 terms). Empty for unbound rows.', readme_font),
    ('TESTCD universe widening: yes — Test_Identity uses the full SDTM_Test_Identity', readme_font),
    ('  universe scoped to measurement domains, not just COSMoS-pinned TESTCDs.', readme_font),
    ('Domain class: Observation_Class joined per Measurement_Specs row.', readme_font),
    ('', None),
    ('CONFLICT FLAGS', section_font),
    ('NCIt_Code_Conflict (Test_Identity) — multiple NCIt concepts resolve from the', readme_font),
    ('  same TESTCD across the graph.', readme_font),
    ('NCIt_Reference_Disagree (Test_Identity) — NCIt at TESTCD grain disagrees', readme_font),
    ('  between graph (COSMoS pin) and reference (SDTM_Test_Identity / NCI EVS).', readme_font),
    ('Validation checks (e.g. TESTCD_NCIt vs BC NCIt_Code, Quantitative-without-units)', readme_font),
    ('  belong in a separate validation step, not in this consumer projection.', readme_font),
    ('  See cosmos-bc-dss/notebooks/COSMoS_BC_DSS_Validate.ipynb for the legacy QC', readme_font),
    ('  pattern; a graph-fed equivalent is planned.', readme_font),
    ('', None),
    ('HEADER COLOUR CONVENTION', section_font),
    ('Green  = TESTCD / SDTM-CT-side columns (NCI EVS reference identity).', readme_font),
    ('Yellow = COSMoS-side columns (BC, DSS, slot pivot — pin and value_list).', readme_font),
    ('Grey   = keys, conflict flags, aggregation columns.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Measurement sub-type — second build of the graph-fed Findings consumer.', readme_font),
    ('Parallel to the legacy sdtm-findings/Measurement_Findings.xlsx, which reads', readme_font),
    ('cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx. The legacy retires once all three', readme_font),
    ('sub-types (specimen, measurement, instrument) are built here.', readme_font),
    ('Not an official CDISC product.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f"ReadMe: {len(readme_lines)} lines")

ReadMe: 62 lines


In [15]:
# ── Test_Identity sheet ──
ws_ti = wb.create_sheet('Test_Identity')

TI_FILLS = {
    'TESTCD':                 GREY_HEADER,
    'NCIt_Code':              GREY_HEADER,
    'In_Scope_Domains':       GREY_HEADER,
    'SDTM_Domains':           GREY_HEADER,
    'TEST':                   GREEN_HEADER,
    'NCIt_Preferred_Term':    GREEN_HEADER,
    'NCIt_Synonyms':          GREEN_HEADER,
    'NCIt_Definition':        GREEN_HEADER,
    'UMLS_CUI':               GREEN_HEADER,
    'NCIm_CUI':               GREEN_HEADER,
    'Has_DSS':                YELLOW_HEADER,
    'DSS_Count':              YELLOW_HEADER,
    'BC_Count':               YELLOW_HEADER,
    'DS_Codes':               YELLOW_HEADER,
    'BC_IDs':                 YELLOW_HEADER,
    'NCIt_Code_Conflict':       GREY_HEADER,
    'NCIt_Reference_Disagree':  GREY_HEADER,
}

TI_WIDTHS = {
    'TESTCD': 14, 'NCIt_Code': 12, 'In_Scope_Domains': 14, 'SDTM_Domains': 18,
    'TEST': 38, 'NCIt_Preferred_Term': 38, 'NCIt_Synonyms': 50,
    'NCIt_Definition': 60, 'UMLS_CUI': 12, 'NCIm_CUI': 12,
    'Has_DSS': 9, 'DSS_Count': 9, 'BC_Count': 9, 'DS_Codes': 30, 'BC_IDs': 30,
    'NCIt_Code_Conflict': 12, 'NCIt_Reference_Disagree': 14,
}

write_sheet(ws_ti, ti_final, TI_FILLS, TI_WIDTHS)
print(f"Test_Identity: {len(ti_final):,} rows x {len(ti_final.columns)} cols")

Test_Identity: 334 rows x 17 cols


In [16]:
# ── Measurement_Specs sheet ──
ws_ms = wb.create_sheet('Measurement_Specs')

# Default everything to yellow; override grey for keys/flags, green for test identity.
MS_FILLS = {c: YELLOW_HEADER for c in ms_final.columns}
MS_FILLS.update({
    'domain': GREY_HEADER, 'Observation_Class': GREY_HEADER,
    'ds_id': GREY_HEADER, 'bc_id': GREY_HEADER, 'TESTCD': GREY_HEADER,
    'TEST': GREEN_HEADER, 'TESTCD_NCIt': GREEN_HEADER,
})

MS_WIDTHS = {
    'domain': 8, 'Observation_Class': 14, 'ds_id': 16, 'bc_id': 14, 'TESTCD': 14,
    'TEST': 38, 'TESTCD_NCIt': 12,
    'bc_short_name': 32, 'bc_definition': 50, 'bc_categories': 35,
    'bc_parent_label': 30, 'bc_hierarchy_path': 45, 'bc_type': 12,
    'result_scales': 18, 'bc_ncit_code': 12,
    'ds_short_name': 38, 'sdtmig_start_version': 12, 'sdtmig_end_version': 12,
    'package_date': 12, 'source': 16,
    'METHOD_value': 22, 'METHOD_ncit': 12, 'METHOD_codelist': 14,
    'ORRESU_value': 12, 'ORRESU_ncit': 12, 'ORRESU_codelist': 14,
    'STRESU_value': 12, 'STRESU_ncit': 12, 'STRESU_codelist': 14,
    'LOC_value': 18, 'LOC_ncit': 12, 'LOC_codelist': 12,
    'EVAL_value': 22, 'EVAL_ncit': 12, 'EVAL_codelist': 14,
    'LOINC_DSS': 14, 'LOINC_BC': 26,
    'Allowed_Units': 50,
}

write_sheet(ws_ms, ms_final, MS_FILLS, MS_WIDTHS)
print(f"Measurement_Specs: {len(ms_final):,} rows x {len(ms_final.columns)} cols")

Measurement_Specs: 128 rows x 66 cols


In [17]:
wb.save(OUTPUT_FILE)
print(f"\nWritten: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB")


Written: /home/claude/work/cdisc-for-ai/sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx
File size: 92 KB


## 6. Summary

In [18]:
print('=== Measurement_Findings summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Test_Identity:    {len(ti_final):>5,} rows x {len(ti_final.columns):>2} cols')
print(f'  COSMoS-pinned (Has_DSS=Yes):  {(ti_final["Has_DSS"] == "Yes").sum():>5,}')
print(f'  Coverage gap (Has_DSS=No):    {(ti_final["Has_DSS"] == "No").sum():>5,}')
print(f'  NCIt_Code_Conflict=Yes:       {(ti_final["NCIt_Code_Conflict"] == "Yes").sum():>5}')
print(f'  NCIt_Reference_Disagree=Yes:  {(ti_final["NCIt_Reference_Disagree"] == "Yes").sum():>5}')
print()
print(f'Measurement_Specs: {len(ms_final):>4,} rows x {len(ms_final.columns):>2} cols')
print(f'  Domains:                     {sorted(ms_final["domain"].unique())}')
print(f'  Slots emitted:                {slots_to_emit}')
print(f'  Allowed_Units expanded:      {(ms_final["Allowed_Units"] != "").sum()} rows')

=== Measurement_Findings summary ===
Output: sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx

Test_Identity:      334 rows x 17 cols
  COSMoS-pinned (Has_DSS=Yes):    123
  Coverage gap (Has_DSS=No):      211
  NCIt_Code_Conflict=Yes:           0
  NCIt_Reference_Disagree=Yes:      0

Measurement_Specs:  128 rows x 66 cols
  Domains:                     ['MK', 'VS']
  Slots emitted:                ['METHOD', 'ORRES', 'STRESC', 'STRESN', 'ORRESU', 'STRESU', 'LOC', 'LAT', 'POS', 'EVAL', 'EVALID']
  Allowed_Units expanded:      7 rows
